# Gold Layer Aggregation Pipeline - Generic & Metadata-Driven

Generic notebook to combine, join, and aggregate Silver tables into Gold business marts based on config.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_path", "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/config/pipeline_config.json", "Config Path")
dbutils.widgets.text("business_domain", "sales", "Business Domain")
dbutils.widgets.text("target_table", "customer_summary", "Target Gold Table")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")

config_path = dbutils.widgets.get("config_path")
business_domain = dbutils.widgets.get("business_domain")
target_table = dbutils.widgets.get("target_table")
environment = dbutils.widgets.get("environment")

print(f"Executing Gold Aggregation for {target_table} ({business_domain} domain, {environment} environment)")

In [ ]:
# Imports & Path setup
import sys
sys.path.append("/Workspace/Users/jayarampogakula@gmail.com/lakeforge")

from lakeforge import (
    ConfigParser,
    GoldAggregator,
    create_trust_engine
)

print("✅ Framework libraries loaded")

In [ ]:
# Load Configurations
config = ConfigParser.parse_monolithic_pipeline_config(config_path)
env_config = config["environments"][environment]

catalog = env_config["catalog"]
silver_schema = env_config["silver_schema"]
gold_schema = env_config["gold_schema"]

gold_config = config["gold_aggregations"][target_table]
print(f"Parsed Gold Config for: {target_table}")
print(gold_config)

In [ ]:
# Load Silver source tables dynamically
silver_dfs = {}
for alias, table_path in gold_config["source_tables"].items():
    # Extract basename, e.g., 'silver.customers_clean' -> 'customers_clean'
    table_basename = table_path.split(".")[-1]
    full_silver_path = f"{catalog}.{silver_schema}.{table_basename}"
    
    print(f"Loading source: {alias} -> {full_silver_path}")
    silver_dfs[alias] = spark.table(full_silver_path)

print(f"✅ Loaded {len(silver_dfs)} source Silver dataframes")

In [ ]:
# Execute aggregations dynamically
result_df = GoldAggregator.aggregate(spark, silver_dfs, gold_config)
print(f"✅ Aggregation complete. Columns: {result_df.columns}")

In [ ]:
# Write to Gold Delta Table
full_gold_table = f"{catalog}.{gold_schema}.{target_table}"
result_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(full_gold_table)

print(f"✅ Wrote {result_df.count()} records to Gold table: {full_gold_table}")

In [ ]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)

# Compare first source table row count with target aggregated row count for reconciliation
first_source = list(silver_dfs.values())[0]
trust_validations = [
    {
        "type": "row_count",
        "params": {
            "source_df": first_source,
            "target_df": result_df,
            "tolerance_percent": 100.0  # Gold aggregations naturally compress row counts, so tolerance is high
        }
    }
]

trust_results = trust_engine.run_trust_validations(trust_validations)
trust_score = trust_engine.calculate_trust_score(
    table_name=full_gold_table,
    dq_results=trust_results,
    pipeline_stage="gold"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")